# HW03-20234080227-申耀翔
## 作业3 - 计算机视觉

丁平尖 2026年6月2日

## 2 卷积和池化层

### 2.1 理论计算题

**已知：**
- 输入：\( 3 \times 32 \times 32 \)（C×H×W）
- 卷积核：16 个，每个 \( 3 \times 5 \times 5 \)
- Padding = 2，Stride = 2

**1. 输出特征图尺寸**

卷积输出尺寸公式：
\[ H_{out} = \left\lfloor \frac{H_{in} + 2P - K}{S} \right\rfloor + 1 \]

\[ H_{out} = \left\lfloor \frac{32 + 4 - 5}{2} \right\rfloor + 1 = 15 + 1 = 16 \]
\[ W_{out} = 16 \]
通道数 = 16

**答案：\(16 \times 16 \times 16\)**

**2. 单个输出通道的一个像素值需要多少次点乘（乘法）**

卷积核大小 = \(3 \times 5 \times 5 = 75\)

**答案：75**

In [ ]:
# 2.2 编程题：二维最大池化前向传播
import numpy as np

def max_pool2d_forward(x, kernel_size, stride, padding):
    """
    x: (batch_size, channels, height, width)
    kernel_size: int (square kernel)
    stride: int
    padding: int
    """
    batch, ch, h, w = x.shape
    pad_x = np.pad(x, ((0,0), (0,0), (padding, padding), (padding, padding)), mode='constant')

    out_h = (h + 2*padding - kernel_size) // stride + 1
    out_w = (w + 2*padding - kernel_size) // stride + 1

    out = np.zeros((batch, ch, out_h, out_w))

    for i in range(out_h):
        for j in range(out_w):
            h_start = i * stride
            h_end = h_start + kernel_size
            w_start = j * stride
            w_end = w_start + kernel_size
            window = pad_x[:, :, h_start:h_end, w_start:w_end]
            out[:, :, i, j] = np.max(window, axis=(2, 3))

    return out

# 测试
x = np.random.randn(2, 3, 32, 32)
y = max_pool2d_forward(x, kernel_size=2, stride=2, padding=0)
print(f"MaxPool2d输出形状: {y.shape}")  # (2, 3, 16, 16)

## 3 LeNet, AlexNet, VGG, NiN

### 3.1 理论计算题

**1. \(5 \times 5\) 卷积层（不带 bias）参数量**

输入通道 = C，输出通道 = C
\[ \text{参数量} = C \times C \times 5 \times 5 = 25C^2 \]

**答案：\(25C^2\)**

**2. 两个串联 \(3 \times 3\) 卷积层（不带 bias）**

第一层：\(9C^2\)，第二层：\(9C^2\)，总和 = \(18C^2\)

**答案：\(18C^2\)**

In [ ]:
# 3.2 编程题：NiN Block
import torch
import torch.nn as nn

class NiNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, 1),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, 1),
            nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)

# 测试
block = NiNBlock(3, 16, kernel_size=3, stride=1, padding=1)
x = torch.randn(1, 3, 32, 32)
y = block(x)
print(f"NiN Block输出形状: {y.shape}")  # torch.Size([1, 16, 32, 32])

## 4 Inception, BN, 残差网络

### 4.1 理论计算题

已知：\(x = [2,4,6,8]\), \(\gamma=2\), \(\beta=1\), \(\epsilon=0\)

均值 \(\mu = 5\)，方差 \(\sigma^2 = 5\)

\[ \hat{x}_i = \frac{x_i - 5}{\sqrt{5}} \]
\[ y_i = 2\hat{x}_i + 1 \]

计算结果：
- \(y_1 \approx -1.68\)
- \(y_2 \approx 0.11\)
- \(y_3 \approx 1.89\)
- \(y_4 \approx 3.68\)

**答案：[-1.68, 0.11, 1.89, 3.68]**

In [ ]:
# 4.2 编程题：Residual Block
class Residual(nn.Module):
    def __init__(self, in_channels, out_channels, use_1x1conv=False, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)

        if use_1x1conv:
            self.shortcut = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride)
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return torch.relu(out + self.shortcut(x))

# 测试
res = Residual(3, 16, use_1x1conv=True, stride=2)
x = torch.randn(1, 3, 32, 32)
y = res(x)
print(f"Residual Block输出形状: {y.shape}")  # torch.Size([1, 16, 16, 16])

## 5 图像增广，微调

### 5.1 理论题

1. **为什么底层小学习率，顶层大学习率？**
   - 底层特征提取器已学到通用特征（边缘、纹理），不应大幅改动
   - 顶层随机初始化，需要快速适应新任务

2. **目标数据集小且相似时的微调策略**
   - 冻结大部分底层，只微调最后几层
   - 使用更小的学习率
   - 使用更强的正则化（dropout、weight decay）
   - 使用数据增广

In [ ]:
# 5.2 编程题：图像增广 Pipeline
from torchvision import transforms

augmentation_pipeline = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.08, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    transforms.ToTensor()
])

print("图像增广Pipeline创建成功")
print(augmentation_pipeline)

## 6 目标检测，训练技巧

### 6.1 理论计算题：IoU

A = [10,10,50,50]，B = [30,30,70,70]

交集面积 = \(20 \times 20 = 400\)

A面积 = \(40 \times 40 = 1600\)，B面积 = 1600

并集面积 = \(1600 + 1600 - 400 = 2800\)

\[ IoU = \frac{400}{2800} = \frac{1}{7} \approx 0.1429 \]

**答案：\(\frac{1}{7}\)**

In [ ]:
# 6.2 编程题：标签平滑交叉熵
import torch.nn.functional as F

def label_smoothing_cross_entropy(logits, labels, epsilon=0.1):
    """
    logits: (batch_size, num_classes)
    labels: (batch_size,) 类别索引
    epsilon: 平滑因子
    """
    num_classes = logits.size(-1)
    log_probs = F.log_softmax(logits, dim=-1)

    # 构造平滑标签
    smooth_labels = torch.full_like(log_probs, epsilon / (num_classes - 1))
    smooth_labels.scatter_(1, labels.unsqueeze(1), 1 - epsilon)

    loss = (-smooth_labels * log_probs).sum(dim=-1).mean()
    return loss

# 测试
logits = torch.randn(4, 10)
labels = torch.tensor([1, 3, 5, 7])
loss = label_smoothing_cross_entropy(logits, labels)
print(f"标签平滑交叉熵损失: {loss.item():.4f}")